In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import pandas as pd
from src.functions import collect_game

COLLECTION_DATE = "2026-11-10"

W1111 17:16:57.173000 28602 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [2]:
import json
import requests

In [ ]:
APP_ID = 3321460      # Crimson Desert
URL = "https://api.steampowered.com/IUserReviewsService/GetAppReviews/v1/"

query = {
    "appid": APP_ID,
    "filter": 1,                 # 1 = Recent: newest reviews first
    "languages": ["english"],
    "num_per_page": 100,         # maximum allowed per request
    "cursor": "*",               # "*" = start at the first page
    "purchase_type": 1,          # 1 = All buyers, not only Steam purchases
}

In [ ]:
response = requests.get(URL, params={"input_json": json.dumps(query)}, timeout=30)
print("HTTP status:", response.status_code)

data = response.json()
payload = data.get("response", data)
print("Top-level keys:", list(payload.keys()))
print("Query summary:", payload.get("query_summary"))

In [ ]:
reviews = payload.get("reviews", [])
print("Reviews on this page:", len(reviews))
print("Next cursor:", payload.get("cursor"))

# Copy the first review and remove the author's SteamID before printing
first = dict(reviews[0])
first["author"] = {k: v for k, v in first["author"].items() if k != "steamid"}
print(json.dumps(first, indent=2))

In [ ]:
for r in reviews[:8]:
    hours = r["author"]["playtime_at_review"] / 60   # playtime is in minutes
    verdict = "👍" if r["voted_up"] else "👎"
    print(f"{verdict} | {hours:.1f} h | {r['review'][:300]!r}\n")

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

from src.functions import fetch_reviews, reviews_to_dataframe

In [ ]:
raw = fetch_reviews(3321460, max_reviews=300)
cd = reviews_to_dataframe(raw, app_id=3321460, game="Crimson Desert")

print(cd.shape)
print("Duplicate review IDs:", cd["review_id"].duplicated().sum())
print("Date range:", cd["created"].min(), "→", cd["created"].max())
print("Share positive:", cd["voted_up"].mean().round(3))
print("Reviews with GPU info:", cd["gpu"].notna().sum())
cd.head()

In [ ]:
import pandas as pd

start = int(pd.Timestamp("2026-03-19", tz="UTC").timestamp())   # launch day
end = int(pd.Timestamp("2026-03-26", tz="UTC").timestamp())     # one week later

raw_launch = fetch_reviews(
    3321460,
    max_reviews=200,
    sort=0,                                  # 0 = Helpful
    extra_filters={"date_range_start": start, "date_range_end": end, "day_range": 0},
)
launch = reviews_to_dataframe(raw_launch, app_id=3321460, game="Crimson Desert")

print(launch.shape)
print("Date range:", launch["created"].min(), "→", launch["created"].max())
print("Share positive:", launch["voted_up"].mean().round(3))
for t in launch["text"].head(5):
    print(repr(t[:250]), "\n")

In [ ]:
launch["edited"] = (launch["updated"] - launch["created"]) > pd.Timedelta(days=1)

print("Reviews edited more than a day later:", launch["edited"].sum())
print(launch.groupby("edited")["voted_up"].agg(["count", "mean"]).round(3))

In [ ]:
launch.groupby(launch["created"].dt.date)["voted_up"].agg(["count", "mean"]).round(3)

In [ ]:
from src.functions import collect_game

COLLECTION_DATE = "2026-11-10"

cd_test = collect_game(3321460, "Crimson Desert", "2026-03-19", COLLECTION_DATE)

print(cd_test.shape)
print(cd_test.groupby("window").agg(
    first=("created", "min"),
    last=("created", "max"),
    n=("review_id", "count"),
    positive=("voted_up", "mean"),
).round(3))

In [ ]:
import time
from src.functions import count_reviews

edges = pd.date_range("2026-03-19", COLLECTION_DATE, periods=6, tz="UTC")
rows = []
for i in range(5):
    start = int(edges[i].timestamp())
    end = int(edges[i + 1].timestamp()) - 1
    rows.append({
        "window": i + 1,
        "positive": count_reviews(3321460, start, end, review_type=1),
        "negative": count_reviews(3321460, start, end, review_type=2),
    })
    time.sleep(1)

real = pd.DataFrame(rows)
real["real_share"] = (real["positive"] / (real["positive"] + real["negative"])).round(3)
real["sample_share"] = cd_test.groupby("window")["voted_up"].mean().round(3).values
print("Total reviews across windows:", real["positive"].sum() + real["negative"].sum())
real

In [ ]:
summary = cd_test.groupby("window").agg(
    first=("created", "min"),
    last=("created", "max"),
    n=("review_id", "count"),
    positive=("voted_up", "mean"),
)
summary["positive"] = summary["positive"].round(3)
summary

In [ ]:
cd_test = collect_game(3321460, "Crimson Desert", "2026-03-19", COLLECTION_DATE)

summary = cd_test.groupby("window").agg(
    n=("review_id", "count"),
    sample_share=("voted_up", "mean"),
    real_share=("window_real_share", "first"),
)
summary["sample_share"] = summary["sample_share"].round(3)
summary

In [ ]:
import time
from src.functions import count_reviews, collect_all, ROOT, RAW

def n_reviews(app_id, start, end):
    """Positive + negative reviews created between two Unix timestamps."""
    return sum((count_reviews(app_id, start, end, review_type=t) or 0) for t in (1, 2))

games = pd.read_csv(ROOT / "data" / "games.csv")
rows = []
for g in games.itertuples():
    release = int(pd.Timestamp(g.steam_release, tz="UTC").timestamp())
    rows.append({
        "game": g.game,
        "steam_release": g.steam_release,
        "before_release": n_reviews(g.app_id, 0, release - 1),
        "first_30_days": n_reviews(g.app_id, release, release + 30 * 86400),
    })
    time.sleep(0.5)

dates_check = pd.DataFrame(rows)
dates_check

## Collect all reviews.

In [ ]:
corpus = collect_all(ROOT / "data" / "games.csv", COLLECTION_DATE, RAW / "steam_reviews")
print(corpus.shape)

In [ ]:
check = corpus.groupby("game").agg(
    n=("review_id", "count"),
    windows=("window", "nunique"),
    sample_share=("voted_up", "mean"),
    first_review=("created", "min"),
    last_review=("created", "max"),
)
check["sample_share"] = check["sample_share"].round(3)
check.sort_values("n")

## Header images for the app

The app's games page shows each game's banner from its Steam store page. Steam's store endpoint `appdetails` returns the image URL; it is called once per game, and the URLs are saved in `data/games.csv`. Only the URLs are stored: the images stay on Steam's servers.

In [3]:
import time
from src.functions import ROOT, fetch_header_image

games = pd.read_csv(ROOT / "data" / "games.csv")
images = []
for app_id in games["app_id"]:
    images.append(fetch_header_image(app_id))
    time.sleep(1)                                 # be polite to Steam's servers
games["header_image"] = images
games.to_csv(ROOT / "data" / "games.csv", index=False)
print(games[["game", "header_image"]].isna().sum())
games[["game", "header_image"]]

game            0
header_image    0
dtype: int64


,game,header_image
0,Crimson Desert,https://shared.akamai.steamstatic.com/store_it...
1,Baldur's Gate 3,https://shared.akamai.steamstatic.com/store_it...
2,Elden Ring,https://shared.akamai.steamstatic.com/store_it...
3,Cyberpunk 2077,https://shared.akamai.steamstatic.com/store_it...
4,Red Dead Redemption 2,https://shared.akamai.steamstatic.com/store_it...
5,Hollow Knight,https://shared.akamai.steamstatic.com/store_it...
6,The Witcher 3: Wild Hunt,https://shared.akamai.steamstatic.com/store_it...
7,Hades,https://shared.akamai.steamstatic.com/store_it...
8,The Elder Scrolls V: Skyrim Special Edition,https://shared.akamai.steamstatic.com/store_it...
9,Sekiro: Shadows Die Twice,https://shared.akamai.steamstatic.com/store_it...
